In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple, Dict

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from scipy.stats import norm

import torch
import torch.nn as nn
import torch.optim as optim

# ---------------------------------------------------------
# 1. Your data: 3D inputs and 1D outputs (maximisation)
# ---------------------------------------------------------
X_raw = np.array([
    [0.17152521, 0.34391687, 0.2487372],
    [0.24211446, 0.64407427, 0.27243281],
    [0.53490572, 0.39850092, 0.17338873],
    [0.49258141, 0.61159319, 0.34017639],
    [0.13462167, 0.21991724, 0.45820622],
    [0.34552327, 0.94135983, 0.26936348],
    [0.15183663, 0.43999062, 0.99088187],
    [0.64550284, 0.39714294, 0.91977134],
    [0.74691195, 0.28419631, 0.22629985],
    [0.17047699, 0.6970324, 0.14916943],
    [0.22054934, 0.29782524, 0.34355534],
    [0.66601366, 0.67198515, 0.2462953],
    [0.04680895, 0.23136024, 0.77061759],
    [0.60009728, 0.72513573, 0.06608864],
    [0.96599485, 0.86111969, 0.56682913],
    [1.065994, 1.041359, 1.090881],  # historical out-of-bounds
    [0.403482, 0.38217, 0.489363],
    [3.98350e-01, 1.00000e-06, 5.43642e-01],
    [0.962851, 0.987386, 0.040875],
    [0.504564, 0.348726, 0.601264],
    [0.265159, 0.286931, 0.413777]
])

y_raw = np.array([
    -0.1121222,  -0.08796286, -0.11141465, -0.03483531, -0.04800758,
    -0.11062091, -0.39892551, -0.11386851, -0.13146061, -0.09418956,
    -0.04694741, -0.10596504, -0.11804826, -0.03637783, -0.05675837,
    -0.769427956661122, -0.03310307977430594, -0.09333459499358941,
    -0.07627377706316849, -0.05678719487656195, -0.03492633073917894
])

DEVICE = torch.device("cpu")  # change to "cuda" if you have a GPU


# ---------------------------------------------------------
# 2. PyTorch MLP model
# ---------------------------------------------------------
class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes=(64, 32), dropout=0.1):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def _train_mlp(
    Xs: np.ndarray,
    ys: np.ndarray,
    hidden: Tuple[int, ...],
    dropout: float,
    lr: float,
    weight_decay: float,
    n_epochs: int,
    seed: int,
    tol: float = 1e-6,
    patience: int = 60,
):
    """Train with simple early stopping on training loss."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MLPRegressorTorch(
        input_dim=Xs.shape[1],
        hidden_sizes=hidden,
        dropout=dropout
    ).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)
    y_tensor = torch.from_numpy(ys.astype(np.float32)).view(-1, 1).to(DEVICE)

    best_loss = float("inf")
    bad = 0

    for _ in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_tensor)
        loss = criterion(preds, y_tensor)
        loss.backward()
        optimizer.step()

        l = float(loss.item())
        if best_loss - l > tol:
            best_loss = l
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    return model


# ---------------------------------------------------------
# 3. MC Dropout surrogate
# ---------------------------------------------------------
@dataclass
class MCDropoutSurrogate:
    hidden_layer_sizes: Tuple[int, ...] = (64, 32)
    dropout: float = 0.1
    n_epochs: int = 1500
    lr: float = 1e-3
    weight_decay: float = 0.0
    random_state: int = 0
    n_mc_samples: int = 200

    def __post_init__(self):
        self.model = None
        self.x_scaler = StandardScaler()
        self.y_scaler = StandardScaler()

    def fit(self, X: np.ndarray, y: np.ndarray):
        Xs = self.x_scaler.fit_transform(X)
        ys = self.y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

        self.model = _train_mlp(
            Xs, ys,
            hidden=self.hidden_layer_sizes,
            dropout=self.dropout,
            lr=self.lr,
            weight_decay=self.weight_decay,
            n_epochs=self.n_epochs,
            seed=self.random_state,
        )

    def predict(self, X: np.ndarray, return_std: bool = False):
        Xs = self.x_scaler.transform(X)
        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)

        # Keep dropout ON at inference for uncertainty
        self.model.train()

        preds_scaled_mc = []
        with torch.no_grad():
            for _ in range(self.n_mc_samples):
                preds_scaled_mc.append(self.model(X_tensor).cpu().numpy().ravel())

        preds_scaled_mc = np.stack(preds_scaled_mc, axis=0)
        mean_scaled = preds_scaled_mc.mean(axis=0)
        std_scaled = preds_scaled_mc.std(axis=0)

        scale_y = self.y_scaler.scale_[0]
        mean_y = self.y_scaler.mean_[0]

        mean = mean_scaled * scale_y + mean_y
        if not return_std:
            return mean

        std = std_scaled * abs(scale_y)
        return mean, std


# ---------------------------------------------------------
# 4. Acquisition helpers (for reporting only)
# ---------------------------------------------------------
def acquisition_pi_ei(mu: np.ndarray, sigma: np.ndarray, y_best: float, xi: float = 0.0):
    sigma = np.maximum(sigma, 1e-6)
    gamma = (mu - y_best - xi) / sigma
    pi = norm.cdf(gamma)
    ei = (mu - y_best - xi) * pi + sigma * norm.pdf(gamma)
    return pi, np.maximum(ei, 0.0)


# ---------------------------------------------------------
# 5. MUST-IMPROVE proposal:
#    - sample ultra-tight micro neighborhood
#    - FILTER: keep only candidates with μ > y_best
#    - choose max μ among those
#    - if none, tighten neighborhood and try again (multi-pass fallback)
# ---------------------------------------------------------
def propose_next_point_must_improve(
    surrogate,
    X_obs: np.ndarray,
    y_obs: np.ndarray,
    random_state: int = 123,
    bounds: Tuple[np.ndarray, np.ndarray] = None,
    passes: Tuple[Dict, ...] = (
        # pass 1: tight, many candidates
        dict(n_candidates=400_000, sigmas=(0.002, 0.004, 0.008), fracs=(0.55, 0.30, 0.15)),
        # pass 2: even tighter (if surrogate is pessimistic)
        dict(n_candidates=400_000, sigmas=(0.001, 0.002, 0.004), fracs=(0.60, 0.30, 0.10)),
        # pass 3: extremely tight (last attempt)
        dict(n_candidates=400_000, sigmas=(0.0005, 0.001, 0.002), fracs=(0.70, 0.25, 0.05)),
    ),
    xi_for_reporting: float = 0.0
) -> Dict:
    rng = np.random.RandomState(random_state)

    d = X_obs.shape[1]
    if bounds is None:
        lower = np.zeros(d, dtype=float)
        upper = np.ones(d, dtype=float)
    else:
        lower, upper = bounds
        lower = np.asarray(lower, dtype=float)
        upper = np.asarray(upper, dtype=float)

    # Current best observed
    best_idx = int(np.argmax(y_obs))
    y_best = float(y_obs[best_idx])
    x_best = np.clip(X_obs[best_idx], lower, upper)

    # Track best fallback overall (in case μ>y_best never happens)
    best_fallback_idx = None
    best_fallback_mu = -np.inf
    best_fallback_sigma = None
    best_fallback_X = None
    best_fallback_note = ""

    chosen_note = ""

    for p_i, p in enumerate(passes, start=1):
        n_candidates = int(p["n_candidates"])
        sigmas = tuple(p["sigmas"])
        fracs = np.asarray(p["fracs"], dtype=float)
        fracs = fracs / fracs.sum()

        counts = (fracs * n_candidates).astype(int)
        counts[0] += (n_candidates - counts.sum())  # fix rounding

        # Build candidates around x_best
        X_parts = []
        for s, n_i in zip(sigmas, counts):
            X_i = x_best + rng.normal(0.0, s, size=(n_i, d))
            X_i = np.clip(X_i, lower, upper)
            X_parts.append(X_i)

        X_cand = np.vstack(X_parts)

        mu, sig = surrogate.predict(X_cand, return_std=True)

        # Update fallback: best μ (even if <= y_best)
        idx_mu = int(np.argmax(mu))
        if float(mu[idx_mu]) > best_fallback_mu:
            best_fallback_mu = float(mu[idx_mu])
            best_fallback_sigma = float(sig[idx_mu])
            best_fallback_X = X_cand
            best_fallback_idx = idx_mu
            best_fallback_note = (
                f"Fallback updated at pass {p_i}: max μ={best_fallback_mu:.6f} "
                f"with sigmas={sigmas}."
            )

        # Hard filter: predicted improvement
        improving = np.where(mu > y_best)[0]
        if improving.size > 0:
            best_imp_idx = int(improving[np.argmax(mu[improving])])
            chosen_note = (
                f"Pass {p_i}: Found {improving.size} candidates with μ > y_best. "
                f"Selected max μ among them. sigmas={sigmas}."
            )
            next_idx = best_imp_idx
            next_x = X_cand[next_idx]
            next_mu = float(mu[next_idx])
            next_sigma = float(sig[next_idx])
            pi_all, ei_all = acquisition_pi_ei(mu, sig, y_best=y_best, xi=xi_for_reporting)

            # Nearest neighbors (distance computed with X clipped to bounds)
            X_obs_for_dist = np.clip(X_obs, lower, upper)
            dists = np.linalg.norm(X_obs_for_dist - next_x, axis=1)
            nn_order = np.argsort(dists)[:3]

            reasoning_lines = [
                f"• Bounds used: lower={lower.round(3)}, upper={upper.round(3)}.",
                "• Strategy: MUST-IMPROVE → micro-neighborhood search + filter (μ > y_best) + pick max μ.",
                f"• Decision: {chosen_note}",
                f"• At x_next: μ={next_mu:.6f}, σ={next_sigma:.6f}.",
                f"• Relative to y_best={y_best:.6f}: PI={pi_all[next_idx]:.3f}, EI={ei_all[next_idx]:.6f} (xi={xi_for_reporting}).",
                "• Nearest previously tested points (distance computed with X clipped to [0,1]):"
            ]
            for rank, idx in enumerate(nn_order, start=1):
                reasoning_lines.append(
                    f"   #{rank}: x={X_obs[idx].round(4)}, y={y_obs[idx]:.6f}, dist={dists[idx]:.4f}"
                )

            return dict(
                next_x=next_x,
                pred_mean=next_mu,
                pred_std=next_sigma,
                pi=float(pi_all[next_idx]),
                ei=float(ei_all[next_idx]),
                y_best=y_best,
                x_best=X_obs[best_idx],
                reasoning="\n".join(reasoning_lines),
            )

    # If we reach here, no μ > y_best found in any pass -> return best fallback
    next_idx = int(best_fallback_idx)
    next_x = best_fallback_X[next_idx]
    next_mu = float(best_fallback_mu)
    next_sigma = float(best_fallback_sigma)

    # For reporting PI/EI at fallback
    mu_all, sig_all = surrogate.predict(best_fallback_X, return_std=True)
    pi_all, ei_all = acquisition_pi_ei(mu_all, sig_all, y_best=y_best, xi=xi_for_reporting)

    X_obs_for_dist = np.clip(X_obs, lower, upper)
    dists = np.linalg.norm(X_obs_for_dist - next_x, axis=1)
    nn_order = np.argsort(dists)[:3]

    reasoning_lines = [
        f"• Bounds used: lower={lower.round(3)}, upper={upper.round(3)}.",
        "• Strategy: MUST-IMPROVE → micro-neighborhood search + filter (μ > y_best) + pick max μ.",
        "• Result: No candidates satisfied μ > y_best across all passes.",
        f"• Returned fallback: overall max μ found (still ≤ y_best). {best_fallback_note}",
        f"• At x_next: μ={next_mu:.6f}, σ={next_sigma:.6f}.",
        f"• Relative to y_best={y_best:.6f}: PI={pi_all[next_idx]:.3f}, EI={ei_all[next_idx]:.6f} (xi={xi_for_reporting}).",
        "• Nearest previously tested points (distance computed with X clipped to [0,1]):"
    ]
    for rank, idx in enumerate(nn_order, start=1):
        reasoning_lines.append(
            f"   #{rank}: x={X_obs[idx].round(4)}, y={y_obs[idx]:.6f}, dist={dists[idx]:.4f}"
        )

    return dict(
        next_x=next_x,
        pred_mean=next_mu,
        pred_std=next_sigma,
        pi=float(pi_all[next_idx]),
        ei=float(ei_all[next_idx]),
        y_best=y_best,
        x_best=X_obs[best_idx],
        reasoning="\n".join(reasoning_lines),
    )


# ---------------------------------------------------------
# 6. Hyperparameter tuning (Random Search + Successive Halving)
# ---------------------------------------------------------
def cv_mse_score(
    config: Dict,
    X: np.ndarray,
    y: np.ndarray,
    k: int = 3,
    seed: int = 0
) -> float:
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    mses = []

    for tr_idx, va_idx in kf.split(X):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        surr = MCDropoutSurrogate(
            hidden_layer_sizes=config["hidden"],
            dropout=config["dropout"],
            n_epochs=config["epochs"],
            lr=config["lr"],
            weight_decay=config["weight_decay"],
            random_state=seed,
            n_mc_samples=config["n_mc_samples"],
        )

        surr.fit(Xtr, ytr)
        preds = surr.predict(Xva)
        mses.append(np.mean((preds - yva) ** 2))

    return float(np.mean(mses))


def tune_hyperparameters(
    X: np.ndarray,
    y: np.ndarray,
    random_state: int = 7
) -> Dict:
    rng = np.random.RandomState(random_state)

    hidden_options = [(32, 32), (64, 32), (64, 64), (128, 64)]
    dropout_options = [0.05, 0.10, 0.15, 0.20]  # keep >0 for MC Dropout
    lr_options = [3e-4, 1e-3, 3e-3, 1e-2]
    wd_options = [0.0, 1e-6, 1e-5, 1e-4]

    n_initial = 18
    configs = []
    for _ in range(n_initial):
        cfg = dict(
            hidden=hidden_options[rng.randint(len(hidden_options))],
            dropout=float(dropout_options[rng.randint(len(dropout_options))]),
            lr=float(lr_options[rng.randint(len(lr_options))]),
            weight_decay=float(wd_options[rng.randint(len(wd_options))]),
            n_mc_samples=int([100, 150, 200][rng.randint(3)]),
        )
        configs.append(cfg)

    stage_epochs = [400, 1000, 2000]
    keep_fracs = [0.5, 0.4, 0.25]

    best_overall = None

    for stage, (epochs, keep_frac) in enumerate(zip(stage_epochs, keep_fracs), start=1):
        scored = []
        for cfg in configs:
            cfg_stage = dict(cfg)
            cfg_stage["epochs"] = epochs
            mse = cv_mse_score(cfg_stage, X, y, k=3, seed=0)
            scored.append((mse, cfg_stage))

        scored.sort(key=lambda t: t[0])
        if best_overall is None or scored[0][0] < best_overall[0]:
            best_overall = scored[0]

        k_keep = max(4, int(len(scored) * keep_frac))
        configs = [cfg for _, cfg in scored[:k_keep]]

        print(f"\n--- TUNING STAGE {stage} ---")
        print(f"epochs={epochs}, kept={k_keep}/{len(scored)}")
        print(f"best CV-MSE so far: {best_overall[0]:.6f}")
        print(f"best config so far: {best_overall[1]}")

    return dict(best_cv_mse=best_overall[0], best_config=best_overall[1])


# ---------------------------------------------------------
# 7. Main: tune -> fit -> propose next (must-improve filter)
# ---------------------------------------------------------
def main():
    np.random.seed(0)
    torch.manual_seed(0)

    # Tune hyperparameters
    tuning = tune_hyperparameters(X_raw, y_raw, random_state=7)
    best_cfg = tuning["best_config"]

    # Fit best surrogate on all data
    surrogate = MCDropoutSurrogate(
        hidden_layer_sizes=best_cfg["hidden"],
        dropout=best_cfg["dropout"],
        n_epochs=best_cfg["epochs"],
        lr=best_cfg["lr"],
        weight_decay=best_cfg["weight_decay"],
        random_state=0,
        n_mc_samples=best_cfg["n_mc_samples"],
    )
    surrogate.fit(X_raw, y_raw)

    # Current best observed
    best_idx = int(np.argmax(y_raw))
    current_best_x = X_raw[best_idx]
    current_best_y = float(y_raw[best_idx])

    # Propose next query within [0,1]^3 with must-improve filter + multi-pass tightening
    suggestion = propose_next_point_must_improve(
        surrogate=surrogate,
        X_obs=X_raw,
        y_obs=y_raw,
        random_state=123,
        bounds=(np.zeros(3), np.ones(3)),
        xi_for_reporting=0.0
    )

    print("\n================================================")
    print("WEEK 7 FUNCTION 3 — HYPERPARAMETER TUNING RESULT")
    print("================================================")
    print("Surrogate: mc_dropout")
    print(f"Best CV-MSE (lower is better): {tuning['best_cv_mse']:.6f}")
    print("Best tuned hyperparameters:")
    print(f"  hidden: {best_cfg['hidden']}")
    print(f"  dropout: {best_cfg['dropout']}")
    print(f"  lr: {best_cfg['lr']}")
    print(f"  weight_decay: {best_cfg['weight_decay']}")
    print(f"  n_mc_samples: {best_cfg['n_mc_samples']}")
    print(f"  epochs: {best_cfg['epochs']}")

    print("\n================================================")
    print("CURRENT BEST OBSERVED")
    print("================================================")
    print(f"x_best = {current_best_x}, y_best = {current_best_y:.6f}")

    print("\n================================================")
    print("RECOMMENDED NEXT POINT (must-improve filter)")
    print("================================================")
    print(f"x_next     = {suggestion['next_x']}")
    print(f"μ(x_next)  = {suggestion['pred_mean']:.6f}")
    print(f"σ(x_next)  = {suggestion['pred_std']:.6f}")
    print(f"PI         = {suggestion['pi']:.4f}")
    print(f"EI         = {suggestion['ei']:.6f}")

    print("\n================================================")
    print("REASONING")
    print("================================================")
    print(suggestion["reasoning"])


if __name__ == "__main__":
    main()



--- TUNING STAGE 1 ---
epochs=400, kept=9/18
best CV-MSE so far: 0.020836
best config so far: {'hidden': (64, 64), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0, 'n_mc_samples': 100, 'epochs': 400}

--- TUNING STAGE 2 ---
epochs=1000, kept=4/9
best CV-MSE so far: 0.020691
best config so far: {'hidden': (64, 64), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0, 'n_mc_samples': 100, 'epochs': 1000}

--- TUNING STAGE 3 ---
epochs=2000, kept=4/4
best CV-MSE so far: 0.020691
best config so far: {'hidden': (64, 64), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0, 'n_mc_samples': 100, 'epochs': 1000}

WEEK 7 FUNCTION 3 — HYPERPARAMETER TUNING RESULT
Surrogate: mc_dropout
Best CV-MSE (lower is better): 0.020691
Best tuned hyperparameters:
  hidden: (64, 64)
  dropout: 0.2
  lr: 0.001
  weight_decay: 0.0
  n_mc_samples: 100
  epochs: 1000

CURRENT BEST OBSERVED
x_best = [0.403482 0.38217  0.489363], y_best = -0.033103

RECOMMENDED NEXT POINT (must-improve filter)
x_next     = [0.40375586